In [9]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Loading user_item matrix
rat_mat = np.load("../transformed_Data/user_item_matrix.npy")
# Loading user indices correspondance
user_corres = np.load("../transformed_Data/user_corres_matrix_index.npy")
# Loading movie indices correspondance
movie_corres = np.load("../transformed_Data/movie_corres_matrix_index.npy")
# loading correspondance between movie indices and movie titles
movies_titles = pd.read_csv("../transformed_Data/MovieID_title.csv")
# loading ratings
ratings = np.load("../transformed_Data/ratings_np.npy")

In [11]:
# sparsity of the ratings matrix:
sparsity = 1.0 - ( np.count_nonzero(rat_mat) / float(rat_mat.size) )
print(sparsity)

0.9431750381059136


### 1. Computation of cos similarity matrix and predictions

#### 1.1 Prediction function

In [12]:
#cos similarity between user and items
from sklearn.metrics.pairwise import pairwise_distances
user_similarity = pairwise_distances(rat_mat, metric='cosine')
item_similarity = pairwise_distances(rat_mat.T, metric='cosine')

In [ ]:
def predict(ratings, similarity, type='user'):

    n_users = len(ratings[:,0])
    n_items = len(ratings[:,1])

    if type == 'user':
        
        mean_user_rating = ratings.mean(axis=1)#mean for each user (type = float)
        #np.newaxis to convert mean_user_rating from 1D array of float to 2D array in order to user it with ratings
        #then normalization of ratings var (rating - E)
        ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) #(type === 2D array like rating var)
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
       # pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)]).T 

    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)]) 
        
    x = np.zeros((n_users, n_items))
    for i in range(0,n_items):
        a=max(pred[:,i])
        b=min(pred[:,i])
        c=0
        d=5
        for j in range(0,n_users):
            x[j,i]=(pred[:,i][j]-(a-c))*d/(b-a+c)
    
    return x

In [21]:
user_prediction = predict(rat_mat, user_similarity, type='user')

#### 1.2 transformation of id to movie title and item based recommandations

In [28]:
movie_corres.shape

(3255, 2)

In [29]:
movie_corres[:10, :]

array([[ 0.,  1.],
       [ 1.,  2.],
       [ 2.,  3.],
       [ 3.,  4.],
       [ 4.,  5.],
       [ 5.,  6.],
       [ 6.,  7.],
       [ 7.,  8.],
       [ 8.,  9.],
       [ 9., 10.]])

In [30]:
movies_titles.shape

(3883, 2)

In [31]:
movies_titles.head(10)

,MovieID,title
0,1,Toy Story
1,2,Jumanji
2,3,Grumpier Old Men
3,4,Waiting to Exhale
4,5,Father of the Bride Part II
5,6,Heat
6,7,Sabrina
7,8,Tom and Huck
8,9,Sudden Death
9,10,GoldenEye


In [27]:
item_prediction = predict(rat_mat, item_similarity, type='item')